In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("storage_acount", ".", "Adls")
storage_acount = dbutils.widgets.get("storage_acount")
dbutils.widgets.text("container", ".", "container")
container = dbutils.widgets.get("container")
dbutils.widgets.text("catalog", ".", "catalog")
catalog = dbutils.widgets.get("catalog")
dbutils.widgets.text("container_s", ".", "container_s")
container_s = dbutils.widgets.get("container_s")


In [0]:
abfs = f"abfss://{container}@{storage_acount}.dfs.core.windows.net/"

In [0]:
customers_bz = spark.table(f"{catalog}.{container}.customers").dropDuplicates(["customer_id"])
loans_bz     = spark.table(f"{catalog}.{container}.loans")
payments_bz  = spark.table(f"{catalog}.{container}.payments")

In [0]:
customers_sv = (
    customers_bz
    .withColumn("date_of_birth", F.to_date("date_of_birth"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .filter("customer_id IS NOT NULL AND first_name IS NOT NULL AND last_name IS NOT NULL")
)


In [0]:
loans_sv = loans_bz.filter("loan_amount > 0 AND interest_rate >= 0")

In [0]:
payments_sv = payments_bz.filter("payment_amount >= 0")
ustomers_sv = customers_sv.withColumn("first_name", F.trim(F.initcap("first_name"))) \
                           .withColumn("last_name", F.trim(F.initcap("last_name")))
customers_sv = customers_sv.filter(F.col("email").rlike(r"^[^@]+@[^@]+\.[^@]+$"))
customers_sv = customers_sv.withColumn(
    "age",
    F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25)
)
loans_sv = loans_sv.withColumn(
    "term_months", 
    F.months_between(F.col("end_date"), F.col("start_date"))
)
payments_sv = payments_sv.join(
    loans_sv.select("loan_id"),
    on="loan_id",
    how="inner"
)

In [0]:
customers_sv = customers_sv.withColumn(
    "flag_missing_phone",
    F.when(F.col("phone_number").isNull(), 1).otherwise(0)
)

In [0]:
payments_agg = payments_sv.groupBy("loan_id").agg(F.sum("payment_amount").alias("total_paid"))
loans_sv = loans_sv.join(payments_agg, on="loan_id", how="left") \
                   .withColumn("balance_due", F.col("loan_amount") - F.col("total_paid"))

In [0]:
payments_agg = payments_sv.groupBy("loan_id").agg(
    F.sum("payment_amount").alias("total_paid_agg"),
    F.max("payment_date").alias("last_payment_date")
)

# Join con loans
loans_with_payments = loans_sv.join(
    payments_agg,
    on="loan_id",
    how="left"
)

# Calcular balance usando el nombre único
loans_with_payments = loans_with_payments.withColumn(
    "balance_due", F.col("loan_amount") - F.col("total_paid_agg")
)
customer_loan_summary = customers_sv.join(
    loans_with_payments,
    on="customer_id",
    how="left"
)


In [0]:

customers_sv.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{container_s}.customers")

loans_with_payments.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{container_s}.loans")

payments_sv.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{container_s}.payments")

customer_loan_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{container_s}.customer_loan_summary")